# 06 -- Backtest (with costs)

Vectorised backtest of the portfolio with a simple turnover-based cost model. **Phase 3** replaces this with the event-driven engine (slippage, market impact, partial fills).

In [ ]:
# Parameters (papermill-overridable: `papermill ... -p SYMBOLS '["AAPL","MSFT"]'`)
SYMBOLS = ["ALPHA", "BRAVO", "CHARLIE"]
START = "2018-01-01"
N_DAYS = 600
SEED = 7
USE_SYNTHETIC = True  # set False to fetch real data via core_trading.data.sources


In [ ]:
import numpy as np
import pandas as pd

from core_trading.research.reproducibility import set_seeds

set_seeds(SEED)


def synthetic_bars(symbols, n, seed, start=START):
    """Seeded OHLCV frame in the canonical (symbol, timestamp) layout."""
    rng = np.random.default_rng(seed)
    idx = pd.date_range(start, periods=n, freq="B", tz="UTC")
    frames = []
    for k, sym in enumerate(symbols):
        drift = 0.0003 * (1 + k)
        px = 100.0 + np.cumsum(rng.standard_normal(n) + drift)
        px = np.maximum(px, 1.0)
        high = px + np.abs(rng.standard_normal(n)) * 0.4
        low = px - np.abs(rng.standard_normal(n)) * 0.4
        frame = pd.DataFrame(
            {
                "open": px,
                "high": np.maximum(high, px),
                "low": np.minimum(low, px),
                "close": px,
                "volume": rng.uniform(1e6, 5e6, n),
                "source": "synthetic",
            },
            index=pd.MultiIndex.from_product(
                [[sym], idx], names=["symbol", "timestamp"]
            ),
        )
        frames.append(frame)
    return pd.concat(frames).sort_index()


if USE_SYNTHETIC:
    bars = synthetic_bars(SYMBOLS, N_DAYS, SEED)
else:  # pragma: no cover - exercised only against live vendors
    import asyncio

    from core_trading.data.bars import BarRequest, BarResolution
    from core_trading.data.sources.yfinance_source import YFinanceBarSource

    req = BarRequest(
        symbols=tuple(SYMBOLS),
        resolution=BarResolution.DAY_1,
        start=pd.Timestamp(START, tz="UTC").to_pydatetime(),
        end=pd.Timestamp.now(tz="UTC").to_pydatetime(),
    )
    bars = asyncio.run(YFinanceBarSource().fetch_bars(req))

print(f"loaded {bars.shape[0]} bars across {len(SYMBOLS)} symbols")
bars.head()


In [ ]:
from core_trading.research.feature_store import default_feature_store

store = default_feature_store()
z = store.compute(bars, ['zscore_20'])['zscore_20']
signal = (-z / 2.0).clip(-1.0, 1.0)
weights = signal.unstack('symbol')
gross = weights.abs().sum(axis=1).replace(0.0, np.nan)
weights = weights.div(gross, axis=0).fillna(0.0)

rets = bars['close'].unstack('symbol').pct_change()
rets, weights = rets.align(weights, join='inner')

COST_BPS = 1.0  # round-trip cost per unit turnover, in basis points
turnover = weights.diff().abs().sum(axis=1).fillna(0.0)
# Yesterday's weights earn today's return; charge cost on rebalancing.
gross_ret = (weights.shift(1) * rets).sum(axis=1)
net_ret = gross_ret - turnover * COST_BPS / 1e4
equity = (1.0 + net_ret).cumprod()
print('final equity (net):', round(float(equity.iloc[-1]), 4))
print('avg daily turnover:', round(float(turnover.mean()), 4))

In [ ]:
from core_trading.research.overfitting import sharpe_ratio

print('net Sharpe (ann.):', round(sharpe_ratio(net_ret.dropna()), 3))
dd = equity / equity.cummax() - 1.0
print('max drawdown:', round(float(dd.min()), 4))
equity.plot(title='Equity curve (net of costs)')

**Gate:** positive net Sharpe after realistic costs -> proceed to `07_robustness.ipynb`.